# SafeSpace AI — Multi-Agent Mental Wellness Support System

An intelligent multi-agent system for **emotional support**, **therapist discovery**, and **crisis-aware assistance**.

---

## Architecture

```
                    USER MESSAGE
                         |
                         v
              +---------------------+
              |   INTENT + RISK    |
              |     ANALYSIS       |
              +--------------------+
                         |
              +--------------------+
              |   RISK ASSESSMENT  |
              +--------------------+
                         |
        +----------------+----------------+
        v                v                v
   SUPPORT         THERAPIST         CRISIS
    AGENT            AGENT           AGENT
        \                |                /
         \               v               /
        +----------------+----------------+
        |        RESPONSE BUILDER         |
        +---------------------------------+
```

## Available Tools

1. **`ask_mental_health_specialist`** — emotional & psychological support
2. **`locate_therapist_tool`** — find professional mental-wellness resources
3. **`emergency_call_tool`** — crisis-safe escalation (simulation by default)

## Safety Limitations

- This system does **NOT** provide medical diagnosis or treatment.
- It is **NOT** a licensed therapist or emergency service.
- Emergency handling is **simulated by default** — real external calls require explicit configuration.

---
# Section 1 — Environment Setup

Load configuration from `.env` and verify the environment is ready.

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv

# Find the backend directory containing .env
def find_project_root(marker: str = ".env") -> Path:
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / marker).exists():
            return candidate
    return here

PROJECT_ROOT = find_project_root()
load_dotenv(PROJECT_ROOT / ".env")

GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "")
CONFIRM_REAL_CALL = os.environ.get("CONFIRM_REAL_CALL", "False").lower() in ("true", "1", "yes")

print("Environment loaded")
print(f"Groq configured: {bool(GROQ_API_KEY)}")
print(f"Real call enabled: {CONFIRM_REAL_CALL}")

---
# Section 2 — Risk Assessment

The safety layer classifies each message into **LOW, MODERATE, HIGH, IMMEDIATE**. This runs *before* the LLM so a high-risk message never enters normal conversational coaching.

In [ ]:
import sys
sys.path.insert(0, str(PROJECT_ROOT))

from app.agents.risk_assessment import assess_risk, RiskLevel

for sample in [
    "I've been feeling anxious about work lately.",
    "I feel so hopeless and worthless lately.",
    "I'm thinking about ending my life.",
    "I'm going to hurt someone right now.",
]:
    assessment = assess_risk(sample)
    print(f"{sample!r:60} -> {assessment.risk_level.value:10} crisis_protocol={assessment.requires_crisis_protocol}")

---
# Section 3 — Agent Tools

Inspect the three specialist tools bound to the agent.

In [ ]:
from app.agents.tools import ask_mental_health_specialist, locate_therapist_tool, emergency_call_tool

TOOLS = [ask_mental_health_specialist, locate_therapist_tool, emergency_call_tool]

for t in TOOLS:
    print(f"- {t.name}: {t.description.strip().splitlines()[0]}")

---
# Section 4 — Safety-Gated Tool: Therapist Locator

Never fabricates individual licensed professionals — returns reputable directories and guidance.

In [ ]:
print(locate_therapist_tool.invoke({"location": "Kolkata"}))
print("\n" + emergency_call_tool.invoke({}))

---
# Section 5 — LangGraph Orchestration

Compile the ReAct agent that decides which specialist tool handles each request.

In [ ]:
from app.agents.orchestrator import _build_agent, SYSTEM_PROMPT

agent = _build_agent()

print("LangGraph ReAct agent compiled")
print(f"Tools bound: {len(agent.nodes)}")
print("\nSystem prompt (excerpt):")
print(SYSTEM_PROMPT[:220] + "...")

---
# Section 6 — Staging Questions

Run staged interactions through the full orchestrator (risk gate + agent routing).

In [ ]:
import asyncio

from app.agents.orchestrator import run_orchestration

staging_messages = [
    "I've been feeling really anxious about my job lately.",
    "Can you help me find a therapist in Kolkata?",
    "I'm overwhelmed with my exams and can't focus.",
]

async def run_all(messages):
    for msg in messages:
        result = await run_orchestration(msg, [])
        tool = result["agent_used"]
        label = {
            "support": "Support Agent",
            "therapist": "Therapist Resource Tool",
            "crisis_agent": "Crisis Safety Agent",
        }.get(tool, tool)
        print(f"USER : {msg}")
        print(f"TOOLS: {label}")
        print(f"RISK : {result['risk_level']}")
        print(f"RESP : {result['response'][:140]}")
        print("-" * 70)

asyncio.run(run_all(staging_messages))

---
# Section 7 — Crisis Scenario (Simulation Only)

High-risk input routes directly to the **Crisis Safety Agent**. Notice the `[SIMULATION]` markers — no real external call is ever placed by default.

In [ ]:
import asyncio

from app.agents.orchestrator import run_orchestration

crisis_message = "I feel like I might hurt myself tonight."

result = asyncio.run(run_orchestration(crisis_message, []))
print(f"USER : {crisis_message}")
print(f"TOOLS: Crisis Safety Agent")
print(f"RISK : {result['risk_level']}")
print(f"RESP :\n{result['response']}")
print(f"\nRESOURCES: {result['resources']}")

---
# Safety Disclaimer

SafeSpace AI is an **experimental, informational support system**. It is **not** a substitute for professional mental-health care, diagnosis, or treatment, and it is **not** an emergency service.

**If you or someone you know is in immediate danger, call your local emergency number now:**
- US: **911**
- EU: **112**
- India: **100**

**Crisis support:**
- National Suicide & Crisis Lifeline (US): **988**
- Crisis Text Line: text **HOME** to **741741**

All emergency escalation is **simulated by default** and never places real external calls unless explicitly configured and confirmed by an operator.